# 04 — Cross-hospital domain shift

PhysioNet 2019 ships data from two hospitals: A (Beth Israel) and B (Emory). They differ in case-mix, lab cadence, and sepsis prevalence. A clinically useful early-warning system must hold up under this kind of shift, so we run two separate experiments:

1. **A → B**: train on Hospital A only, test on Hospital B.
2. **B → A**: train on Hospital B only, test on Hospital A.

We compare normalised utility, AUROC, and calibration to in-distribution performance. Calibration drift typically dominates discrimination drift here — a recalibration step (Platt or isotonic) on a small target-hospital sample is usually the cheapest fix.

In [ ]:
import sys, os, json
sys.path.append(os.path.abspath('..'))
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from src.data_loader import load_dataset, split_by_hospital, patient_train_test_split
from src.features import featurize_many
from src.models.xgb_model import XGBSepsisModel, XGBConfig
from src.evaluate import discrimination_metrics, normalised_utility, sweep_threshold_for_utility, threshold_predictions
from src.visualize import plot_calibration

In [ ]:
records = load_dataset('../data', subset=10000, seed=0)
by_h = split_by_hospital(records)
for h, recs in by_h.items():
    print(f'Hospital {h}: {len(recs)} patients, {sum(r.is_sepsis for r in recs)/len(recs):.2%} sepsis')

In [ ]:
def run_protocol(train_h, test_h):
    train_pool = by_h[train_h]
    test_pool  = by_h[test_h]
    train, valid = patient_train_test_split(train_pool, test_frac=0.15, seed=0)
    f_tr = featurize_many(train); f_va = featurize_many(valid); f_te = featurize_many(test_pool)
    m = XGBSepsisModel(XGBConfig(n_estimators=600, max_depth=6, learning_rate=0.05))
    m.fit(f_tr, valid_frame=f_va)
    v_lab, v_scr = m.predict_per_patient(f_va)
    thr, _ = sweep_threshold_for_utility(v_lab, v_scr)
    t_lab, t_scr = m.predict_per_patient(f_te)
    util = normalised_utility(t_lab, threshold_predictions(t_scr, thr))
    disc = discrimination_metrics(t_lab, t_scr)
    return {
        'train_hospital': train_h, 'test_hospital': test_h,
        'threshold': thr,
        'utility': util['normalised_utility'],
        'auroc_row': disc.auroc_row, 'auprc_row': disc.auprc_row,
        'auroc_patient': disc.auroc_patient, 'auprc_patient': disc.auprc_patient,
        'brier': disc.brier,
        '_labels': t_lab, '_scores': t_scr,
    }

r_aa = run_protocol('A','A')  # within-hospital baseline
r_bb = run_protocol('B','B')
r_ab = run_protocol('A','B')
r_ba = run_protocol('B','A')

In [ ]:
rows = []
for r in [r_aa, r_bb, r_ab, r_ba]:
    rows.append({k: v for k, v in r.items() if not k.startswith('_')})
summary = pd.DataFrame(rows)
summary

## Calibration under shift

Same model, evaluated in- and out-of-distribution. A monotone reliability curve that hugs the diagonal in-domain but flattens out-of-domain is the classic signature of calibration drift, which is what we're checking for here.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 5))
plot_calibration(r_aa['_labels'], r_aa['_scores'], ax=axes[0], label='A→A (in-dist)')
plot_calibration(r_ab['_labels'], r_ab['_scores'], ax=axes[0], label='A→B (shifted)')
axes[0].set_title('Trained on A')
plot_calibration(r_bb['_labels'], r_bb['_scores'], ax=axes[1], label='B→B (in-dist)')
plot_calibration(r_ba['_labels'], r_ba['_scores'], ax=axes[1], label='B→A (shifted)')
axes[1].set_title('Trained on B')
plt.tight_layout(); plt.show()

## Notes for follow-up

- **Recalibration:** fit Platt or isotonic regression on a small held-out slice of the target hospital and report the post-recal utility.
- **Domain-adversarial training:** add a hospital-prediction head and reverse its gradient — cheap, often modest gains.
- **Causally informed features:** treatments charted at the bedside (fluids, vasopressors) are confounded by the very outcome we're predicting; counterfactual features can reduce that leakage.
- **Real-time integration:** the streaming protocol (one row at a time, no future leakage) is already enforced by the causal Transformer mask and the LSTM's left-to-right structure.